# Matriz de tiempos en coche: núcleo → servicios

Para cada núcleo de `asturias.gpkg` (capa `settlements`) y cada tipo de servicio, calcula los **K servicios más
cercanos por carretera** y el tiempo en coche hasta cada uno. Es la tabla de la que leerán las herramientas del
agente (`nearest_service`, `filter_settlements`, `simulate_closure`…) y de la que saldrán las respuestas de
referencia del benchmark.

Pasos:
1. Construir la capa `services` (un único esquema para farmacias, centros de salud, hospitales y colegios).
2. Construir la red de carreteras de OSM y ponerle velocidades y tiempos.
3. Enganchar núcleos y servicios al nodo de la red más cercano.
4. Calcular los K más cercanos con Dijkstra y guardar la tabla `access`.
5. Revisar los resultados: tiempos por tipo, núcleos peor servidos y carretera frente a línea recta.

**Antes de ejecutar:** `build_settlements.ipynb` ya ejecutado y el extracto de OSM de Geofabrik
(`asturias-latest.osm.pbf`) en `data/raw/` (ver apartado 2). La red se construye la primera vez y luego se lee de
disco.

In [ ]:
# %pip install -q osmnx scipy osmium

In [ ]:
import glob
import os
import re

import geopandas as gpd
import networkx as nx
import numpy as np
import osmium
import osmnx as ox
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra

RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
GPKG_PATH = os.path.join(PROCESSED_DIR, "asturias.gpkg")
GRAPH_PATH = os.path.join(RAW_DIR, "osm_drive_asturias.graphml")
CRS_METRES = "EPSG:25830"

K = 5                        # nearest services kept per settlement and service type
BORDER_BUFFER_M = 15_000     # roads just outside Asturias (León, Cantabria, Lugo) can be part of the best route
MAX_TRAVEL_S = 3 * 3600      # stop Dijkstra after 3 h: nothing in Asturias should be further
SNAP_SPEED_KMH = 20          # speed for the leg between a point and its nearest road node (tracks, village lanes)

ox.settings.use_cache = True
ox.settings.log_console = False

## 1. Capa de servicios

Cada tipo de servicio es una **pregunta que tiene sentido hacer**, no una capa del SIG. Por eso algunos tipos
juntan varias capas (cualquier punto de atención primaria) y otros filtran (solo hospitales generales).

| `service_type` | Qué incluye | Por qué |
|---|---|---|
| `health_centre` | Centros de salud + consultorios locales de las zonas especiales de salud | La cabecera de cada zona de salud: médico de familia, pediatría, enfermería a diario. Las 16 zonas especiales (concejos de montaña como Somiedo, Ibias o Cabrales) no tienen centro de salud; su atención se reparte entre varios consultorios locales, que hacen de cabecera |
| `primary_care` | Centros de salud + consultorios locales + consultorios periféricos | Cualquier punto donde pasar consulta, aunque sea unos días por semana |
| `hospital` | Los 9 hospitales que la zonificación sanitaria asigna como referencia | Excluye el Instituto Nacional de Silicosis, Monte Naranco, Cruz Roja, Adaro y la Fundación Hospital de Avilés, que no son hospital de referencia de ninguna zona |
| `pharmacy` | Farmacias | |
| `pharmacy_or_kit` | Farmacias + botiquines | El botiquín es una farmacia pequeña dependiente de otra, para pueblos sin farmacia |
| `primary_school` | Colegios públicos de infantil y primaria (CEIP, CP, CPEB) | |
| `secondary_school` | Institutos públicos (IES) y CPEB | Los CPEB (Centros Públicos de Educación Básica) llegan hasta 4.º de ESO |

**Decisión pendiente (Violeta):** colegios públicos solo, o también concertados. Por defecto, solo públicos.
Cambiando `INCLUDE_CONCERTED_SCHOOLS` se añaden los concertados de infantil, primaria y secundaria.

In [ ]:
INCLUDE_CONCERTED_SCHOOLS = False

# Reference hospitals named in the health-zone layer (field hospi_zon), by their id in the hospitals layer.
REFERENCE_HOSPITAL_IDS = {
    110,  # Hospital Universitario Central de Asturias (HUCA)
    103,  # Hospital de Cabueñes
    104,  # Hospital Comarcal de Jarrio
    109,  # Hospital San Agustín
    111,  # Hospital Valle del Nalón
    112,  # Hospital Vital Álvarez Buylla
    107,  # Hospital del Oriente de Asturias Francisco Grande Covián
    102,  # Hospital Carmen y Severo Ochoa
    105,  # Hospital de Jove
}

PRIMARY_SCHOOL_TYPES = {
    "Colegio de Educación Infantil y Primaria",
    "Colegio de Educación Primaria",
    "Centro Público de Educación General Básica",
}
SECONDARY_SCHOOL_TYPES = {
    "Instituto de Educación Secundaria",
    "Centro Público de Educación General Básica",
}
CONCERTED_PRIMARY_TYPES = {"Centro Privado de Educación Infantil Primaria y Secundaria"}
CONCERTED_SECONDARY_TYPES = {"Centro Privado de Educación Infantil Primaria y Secundaria",
                             "Centro Privado de Educación Secundaria"}


def read_raw(pattern):
    """Read the most recent raw GeoJSON whose name contains `pattern`, with lower-case columns."""
    paths = sorted(glob.glob(os.path.join(RAW_DIR, f"*{pattern}*.geojson")))
    if not paths:
        raise FileNotFoundError(f"no raw layer matches '{pattern}' in {RAW_DIR}")
    layer = gpd.read_file(paths[-1])
    layer.columns = [c.lower() for c in layer.columns]
    return layer.to_crs(CRS_METRES)


def to_services(layer, service_type, id_column, name_column, source):
    """Map one raw layer to the common services schema."""
    return gpd.GeoDataFrame({
        "service_id": source + "-" + layer[id_column].astype(str),
        "service_type": service_type,
        "name": layer[name_column].astype(str).str.strip(),
        "source": source,
        "ownership": layer["titular"] if "titular" in layer else None,
    }, geometry=layer.geometry, crs=CRS_METRES)


health_centres = read_raw("centros_de_salud_2")
local_clinics = read_raw("consultorios_locales")
peripheral_clinics = read_raw("consultorios_perifericos")
hospitals = read_raw("hospitales")
pharmacies = read_raw("farmacias_botiquines__farmacias")
kits = read_raw("farmacias_botiquines__botiquines")
schools = read_raw("centros_educativos")

primary_types = PRIMARY_SCHOOL_TYPES | (CONCERTED_PRIMARY_TYPES if INCLUDE_CONCERTED_SCHOOLS else set())
secondary_types = SECONDARY_SCHOOL_TYPES | (CONCERTED_SECONDARY_TYPES if INCLUDE_CONCERTED_SCHOOLS else set())

parts = [
    to_services(health_centres, "health_centre", "sipres", "nombre", "cs"),
    # Special health zones (zona_3 "E…") have no health centre: their local clinics act as the zone head.
    to_services(local_clinics[local_clinics["zona_3"].str.startswith("E")], "health_centre", "sipres", "nombre", "cl"),
    to_services(health_centres, "primary_care", "sipres", "nombre", "cs"),
    to_services(local_clinics, "primary_care", "sipres", "nombre", "cl"),
    # Two pairs of peripheral clinics share a SIPRES code (Grullos/San Román, Villabre/Yernes): use objectid.
    to_services(peripheral_clinics, "primary_care", "objectid", "nombre", "cp"),
    to_services(hospitals[hospitals["id"].isin(REFERENCE_HOSPITAL_IDS)], "hospital", "id", "nombre", "hosp"),
    to_services(pharmacies, "pharmacy", "id_far", "titular", "far"),
    to_services(pharmacies, "pharmacy_or_kit", "id_far", "titular", "far"),
    to_services(kits, "pharmacy_or_kit", "id_bt", "titular", "bot"),
    to_services(schools[schools["tipo"].isin(primary_types)], "primary_school", "codigo", "nombre", "edu"),
    to_services(schools[schools["tipo"].isin(secondary_types)], "secondary_school", "codigo", "nombre", "edu"),
]
services = pd.concat(parts, ignore_index=True)
services = gpd.GeoDataFrame(services, geometry="geometry", crs=CRS_METRES)

# One service can belong to several types (a health centre is also primary care), but within a type ids are unique.
assert not services.duplicated(["service_type", "service_id"]).any(), "repeated service id within a type"
print(services["service_type"].value_counts().to_string())
assert (services["service_type"] == "hospital").sum() == len(REFERENCE_HOSPITAL_IDS), "a reference hospital is missing"

## 2. Red de carreteras (OSM)

Hay dos formas de obtener la red **conducible** (la misma definición que `network_type="drive"` de OSMnx). El
notebook usa la primera que encuentre:

1. **Fichero de Geofabrik (recomendado).** Si en `data/raw/` hay un `*.osm.pbf` (por ejemplo
   `asturias-latest.osm.pbf`, descargado con el navegador desde
   https://download.geofabrik.de/europe/spain/asturias-latest.osm.pbf), la red se construye sin conexión:
   se filtran las vías conducibles con `osmium` y se cargan con `ox.graph_from_xml`. Cubre solo Asturias:
   se pierde el margen por León, Lugo y Cantabria (limitación declarada; afecta a pocas rutas de borde).
   Se pueden poner varios `.pbf` (por ejemplo, también `cantabria-latest.osm.pbf`) y se usan todos.
2. **Overpass.** Si no hay `.pbf`, se descarga con `ox.graph_from_polygon` el contorno de Asturias más un margen
   de 15 km. Puede fallar por límites del servidor o por la red del ordenador.

**Velocidades.** OSMnx usa el `maxspeed` de cada vía cuando existe. Cuando falta, pone la media de las vías del
mismo tipo (`highway`) que sí lo tienen y, si no hay ninguna, el valor de `FALLBACK_SPEEDS_KMH`. Son velocidades
de **flujo libre**: sin tráfico, sin semáforos, sin curvas. En montaña el tiempo real será mayor; se valida en el
apartado 5 y contra los tiempos del MITECO más adelante.

**Componente conexa.** Se queda solo la mayor componente fuertemente conexa (desde cualquier nodo se puede llegar a
cualquier otro respetando los sentidos). Si no, un núcleo enganchado a un trozo aislado no llegaría a nada.

In [ ]:
FALLBACK_SPEEDS_KMH = {
    "motorway": 120, "motorway_link": 60,
    "trunk": 100, "trunk_link": 50,
    "primary": 80, "primary_link": 40,
    "secondary": 70, "secondary_link": 40,
    "tertiary": 50, "tertiary_link": 30,
    "unclassified": 40, "residential": 30, "living_street": 20, "service": 20,
}

# Same rules as OSMnx's "drive" network filter (a value matching the regex excludes the way).
EXCLUDED_HIGHWAY = re.compile(
    r"abandoned|bridleway|bus_guideway|construction|corridor|cycleway|elevator|escalator|footway|no|path|"
    r"pedestrian|planned|platform|proposed|raceway|razed|rest_area|service|services|steps|track")
EXCLUDED_SERVICE = re.compile(r"alley|driveway|emergency_access|parking|parking_aisle|private")


def is_drivable(tags):
    """True if an OSM way with these tags belongs to OSMnx's drive network."""
    highway = tags.get("highway")
    if highway is None or EXCLUDED_HIGHWAY.search(highway):
        return False
    return not (re.search("yes", tags.get("area", ""))
                or re.search("private", tags.get("access", ""))
                or re.search("no", tags.get("motor_vehicle", ""))
                or re.search("no", tags.get("motorcar", ""))
                or EXCLUDED_SERVICE.search(tags.get("service", "")))


def graph_from_pbf(pbf_path):
    """Build the drive network from a Geofabrik .osm.pbf extract, offline."""
    xml_path = pbf_path.replace(".osm.pbf", "_drive.osm")
    kept = 0
    # BackReferenceWriter also writes the nodes that the kept ways reference.
    with osmium.BackReferenceWriter(xml_path, ref_src=pbf_path, overwrite=True) as writer:
        for way in osmium.FileProcessor(pbf_path, osmium.osm.WAY):
            if is_drivable(way.tags):
                writer.add_way(way)
                kept += 1
    print(f"{os.path.basename(pbf_path)}: {kept:,} vías conducibles")
    return ox.graph_from_xml(xml_path, simplify=True)


pbf_paths = sorted(glob.glob(os.path.join(RAW_DIR, "*.osm.pbf")))

if os.path.exists(GRAPH_PATH):
    graph = ox.load_graphml(GRAPH_PATH)
    print("red leída de disco:", GRAPH_PATH)
else:
    if pbf_paths:
        graph = nx.compose_all([graph_from_pbf(path) for path in pbf_paths])
        print("red construida desde:", [os.path.basename(p) for p in pbf_paths])
    else:
        boundary = read_raw("unidadesadministrativas__asturias")
        area = boundary.buffer(BORDER_BUFFER_M).to_crs("EPSG:4326").union_all()
        graph = ox.graph_from_polygon(area, network_type="drive", simplify=True)
        print("red descargada de Overpass")
    graph = ox.add_edge_speeds(graph, hwy_speeds=FALLBACK_SPEEDS_KMH)
    graph = ox.add_edge_travel_times(graph)
    ox.save_graphml(graph, GRAPH_PATH)
    print("red guardada:", GRAPH_PATH)

graph = ox.truncate.largest_component(graph, strongly=True)
graph = ox.project_graph(graph, to_crs=CRS_METRES)
print(f"{graph.number_of_nodes():,} nodos | {graph.number_of_edges():,} tramos")

## 3. Enganchar puntos a la red

Cada núcleo y cada servicio se asocia a su **nodo más cercano** (en línea recta). La distancia hasta ese nodo se
guarda (`snap_m`) y se recorre a `SNAP_SPEED_KMH`: es el trozo de pista o calle que la red no tiene.
Una distancia de enganche grande es una alerta: el punto está mal situado o la red no llega.

In [ ]:
settlements = gpd.read_file(GPKG_PATH, layer="settlements").to_crs(CRS_METRES)


def snap(points):
    """Nearest graph node and straight-line distance to it, for each point."""
    nodes, distances = ox.distance.nearest_nodes(
        graph, X=points.geometry.x.to_numpy(), Y=points.geometry.y.to_numpy(), return_dist=True)
    return np.asarray(nodes), np.asarray(distances)


settlements["node"], settlements["snap_m"] = snap(settlements)
services["node"], services["snap_m"] = snap(services)

for label, points in [("núcleos", settlements), ("servicios", services)]:
    print(f"{label}: distancia al nodo más cercano (m)")
    print(points["snap_m"].describe(percentiles=[.5, .9, .99]).round(0).to_string())
    print()

display(settlements.nlargest(5, "snap_m")[["settlement_id", "name", "population", "snap_m"]])
display(services.nlargest(5, "snap_m")[["service_id", "service_type", "name", "snap_m"]])

## 4. Los K servicios más cercanos por carretera

**Idea.** Interesa el tiempo **del núcleo al servicio**. Calcularlo desde cada núcleo (6.663 Dijkstras) es lento.
En su lugar se lanza un Dijkstra **desde cada servicio sobre la red invertida** (sentidos al revés): la distancia
del servicio a un nodo en la red invertida es la del nodo al servicio en la red real. Así hay tantos Dijkstras como
servicios (unos 1.300) y no como núcleos.

Se usa `scipy.sparse.csgraph.dijkstra`, que está escrito en C y es mucho más rápido que `networkx` para esto.

Tiempo total = enganche del núcleo + carretera + enganche del servicio.

In [ ]:
node_ids = np.array(list(graph.nodes))
node_index = {node: i for i, node in enumerate(node_ids)}

# Sparse matrix of travel times (seconds). With parallel edges, keep the fastest.
edges = ox.convert.graph_to_gdfs(graph, nodes=False)[["travel_time"]].reset_index()
edges = edges.groupby(["u", "v"], as_index=False)["travel_time"].min()
rows = edges["u"].map(node_index).to_numpy()
cols = edges["v"].map(node_index).to_numpy()
road = csr_matrix((edges["travel_time"].to_numpy(), (rows, cols)), shape=(len(node_ids), len(node_ids)))
road_reversed = road.T.tocsr()

settlement_nodes = settlements["node"].map(node_index).to_numpy()
settlement_snap_s = settlements["snap_m"].to_numpy() / (SNAP_SPEED_KMH / 3.6)


def nearest_services(service_points, k=K, batch_size=64):
    """For every settlement, the k fastest services of `service_points`: (service row, seconds) arrays."""
    n_settlements = len(settlement_nodes)
    best_rows = np.full((n_settlements, 0), -1)
    best_times = np.full((n_settlements, 0), np.inf)
    source_nodes = service_points["node"].map(node_index).to_numpy()
    service_snap_s = service_points["snap_m"].to_numpy() / (SNAP_SPEED_KMH / 3.6)

    for start in range(0, len(source_nodes), batch_size):
        batch = slice(start, start + batch_size)
        times = dijkstra(road_reversed, directed=True, indices=source_nodes[batch], limit=MAX_TRAVEL_S)
        # rows: services in batch; columns: settlements. Add both snap legs.
        times = times[:, settlement_nodes] + settlement_snap_s[None, :] + service_snap_s[batch, None]
        batch_rows = np.broadcast_to(np.arange(len(source_nodes))[batch][:, None], times.shape)
        best_times = np.hstack([best_times, times.T])
        best_rows = np.hstack([best_rows, batch_rows.T])
        keep = np.argsort(best_times, axis=1)[:, :k]
        best_times = np.take_along_axis(best_times, keep, axis=1)
        best_rows = np.take_along_axis(best_rows, keep, axis=1)
    return best_rows, best_times


def straight_km(settlement_rows, service_points, service_rows):
    origin = settlements.geometry.iloc[settlement_rows].reset_index(drop=True)
    target = service_points.geometry.iloc[service_rows].reset_index(drop=True)
    return origin.distance(target).to_numpy() / 1000


tables = []
for service_type, group in services.groupby("service_type"):
    group = group.reset_index(drop=True)
    rows_k, times_k = nearest_services(group)
    k_used = rows_k.shape[1]
    settlement_rows = np.repeat(np.arange(len(settlements)), k_used)
    reached = np.isfinite(times_k.ravel())
    table = pd.DataFrame({
        "settlement_id": settlements["settlement_id"].to_numpy()[settlement_rows],
        "service_type": service_type,
        "rank": np.tile(np.arange(1, k_used + 1), len(settlements)),
        "service_id": group["service_id"].to_numpy()[rows_k.ravel()],
        "minutes": times_k.ravel() / 60,
    })[reached]
    table["straight_km"] = straight_km(settlement_rows[reached], group, rows_k.ravel()[reached])
    tables.append(table)
    print(f"{service_type}: {len(group)} servicios, {int((~reached).sum())} pares sin ruta en {MAX_TRAVEL_S // 3600} h")

access = pd.concat(tables, ignore_index=True).round({"minutes": 1, "straight_km": 2})
print(f"\n{len(access):,} filas en la tabla access")
display(access.head(10))

## 5. Revisión de resultados

Tres comprobaciones antes de fiarse de la tabla:
1. **Tiempos al servicio más cercano por tipo.** Órdenes de magnitud razonables: farmacia en minutos, hospital en
   decenas de minutos.
2. **Núcleos habitados peor servidos.** Deberían ser aldeas de montaña conocidas (Somiedo, Ibias, Ponga…), no
   puntos mal situados.
3. **Carretera frente a línea recta.** El cociente km por carretera / km en línea recta suele estar entre 1,2 y 2 en
   zonas rurales; aquí se usa el tiempo, así que se mira la **velocidad equivalente en línea recta**
   (km en línea recta / horas). Valores por encima de ~90 km/h indican un error.

In [ ]:
nearest = access[access["rank"] == 1].merge(
    settlements[["settlement_id", "name", "municipality_code", "population"]], on="settlement_id")

print("Minutos al servicio más cercano, por tipo (todos los núcleos):")
display(nearest.groupby("service_type")["minutes"].describe(percentiles=[.5, .9, .99]).round(1))

inhabited = nearest[nearest["population"] > 0]
print("Población a más de 15 / 30 min del servicio más cercano:")
summary = inhabited.groupby("service_type").apply(lambda g: pd.Series({
    "> 15 min": int(g.loc[g["minutes"] > 15, "population"].sum()),
    "> 30 min": int(g.loc[g["minutes"] > 30, "population"].sum()),
}), include_groups=False)
display(summary)

print("Núcleos con más de 20 habitantes peor servidos para centro de salud:")
display(inhabited[(inhabited["service_type"] == "health_centre") & (inhabited["population"] > 20)]
        .nlargest(10, "minutes")[["name", "municipality_code", "population", "minutes", "straight_km"]])

nearest["straight_kmh"] = nearest["straight_km"] / (nearest["minutes"] / 60)
print("Velocidad equivalente en línea recta (km/h), servicio más cercano:")
print(nearest.loc[nearest["minutes"] > 2, "straight_kmh"].describe(percentiles=[.1, .5, .9, .99]).round(1).to_string())

## 6. Guardar

En `asturias.gpkg`:
- `services`: puntos de servicio con el esquema común (un servicio aparece una vez por cada tipo al que pertenece);
- `access`: tabla sin geometría, una fila por núcleo, tipo de servicio y rango (1 = el más cercano).

La red (`osm_drive_asturias.graphml`) se queda en `data/raw/`. Pesa bastante: conviene no subirla a git
(añadirla a `.gitignore`), porque se regenera ejecutando este notebook.

In [ ]:
services.drop(columns=["node"]).to_file(GPKG_PATH, layer="services", driver="GPKG")
gpd.GeoDataFrame(access).to_file(GPKG_PATH, layer="access", driver="GPKG")
print("capas guardadas en", GPKG_PATH, "->", gpd.list_layers(GPKG_PATH)["name"].tolist())

## Conclusiones (ejecución del 11/10/2026)

Red construida desde el extracto de Geofabrik `asturias-261008.osm.pbf`: 58.587 vías conducibles, que dan 62.025
nodos y 146.641 tramos en la mayor componente fuertemente conexa. Todos los núcleos llegan a todos los tipos de
servicio en menos de 3 h.

1. **Tiempos.** Mediana al servicio más cercano: 5,6 min a farmacia, 5,3 a cualquier punto de atención primaria,
   5,9 a colegio, 8,7 a instituto y 17,6 a hospital de referencia. Máximo, unos 78 min. Son tiempos de flujo libre:
   en montaña se quedan cortos (pendiente de validar con el MITECO).
2. **Centros de salud.** En esa ejecución `health_centre` solo tenía los 69 centros de salud, y los peor servidos
   (Sena, Sotres, Saliencia, Valle de Lago, a 50-60 min) eran de zonas especiales de salud que no tienen centro de
   salud sino consultorios locales. Por eso `health_centre` incluye ahora esos consultorios (tabla del apartado 1);
   hay que volver a ejecutar para tener las cifras nuevas.
3. **Enganche.** Mediana de 66 m para los núcleos y 33 m para los servicios. Los casos grandes son aldeas sin
   carretera (Bulnes, 2,7 km, se sube a pie o en funicular). Arangas (48 hab., 2,6 km) está pendiente de revisar en
   un mapa: tiene carretera, así que el punto o la red pueden estar mal.
4. **Velocidad equivalente en línea recta.** Mediana 32 km/h, percentil 99 de 67 km/h y máximo de 94 km/h, apenas
   por encima del umbral de alerta de 90 km/h; no hay indicios de errores en la red.
5. **Limitación.** El extracto solo cubre Asturias: se pierde el margen de 15 km por León, Lugo y Cantabria, y
   algunas rutas de los concejos de borde pueden salir más largas de lo real.
6. **Zonificación.** Las zonas II.9 (Oviedo-Teatinos) y II.13 (Siero-Lugones) no tienen ningún centro de salud con
   su código en la capa del SIG, y las zonas II.7 y III.2 tienen dos. Parece un error de codificación de la capa; no
   afecta a los tiempos, solo a consultas por zona.
